# Rule-Based Lane Following

This notebook follows the lane using simple geometry instead of a trained model.

Strategy:

- detect the green center tape in a cropped region of the frame
- estimate two center points on that line
- compute lateral offset and heading
- combine them into a steering value
- optionally send one short drive pulse to the rover

This is a good fit for your LEGO loop because the track appearance is simple and repeatable.


In [1]:
from pathlib import Path
import glob
import os
import sys
import time

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

video_devices = sorted(glob.glob('/dev/video*'))
print('Project root:', project_root)
print('Video devices:', video_devices)
print('Basic notebook imports ok')


Project root: /home/orin/JetCar
Video devices: ['/dev/video0', '/dev/video1']
Basic notebook imports ok


In [2]:
import io
import json

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from PIL import Image
import serial
import serial.tools.list_ports

from jetcar.camera import open_camera, read_rgb_frame

camera_handle = None
serial_handle = None
current_frame = None
last_analysis = None


/home/orin/JetCar/.venv/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [3]:
def available_ports():
    ports = [port.device for port in serial.tools.list_ports.comports()]
    for fallback in ('/dev/ttyTHS1', '/dev/ttyTHS2'):
        if os.path.exists(fallback) and fallback not in ports:
            ports.append(fallback)
    return ports


def set_status(text):
    status.value = f'<b>Status:</b> {text}'


def log(message):
    with log_output:
        print(message)


def render_image(widget, frame_rgb):
    image = Image.fromarray(frame_rgb)
    with io.BytesIO() as buf:
        image.save(buf, format='JPEG')
        widget.value = buf.getvalue()


def open_selected_camera():
    global camera_handle
    close_camera()
    camera_handle = open_camera(
        source=camera_source_dropdown.value,
        sensor_id=sensor_id.value,
        device_index=device_index.value,
        width=frame_width.value,
        height=frame_height.value,
        warmup_frames=warmup_frames.value,
    )
    set_status(f'camera open ({camera_source_dropdown.value})')


def close_camera():
    global camera_handle
    if camera_handle is not None:
        camera_handle.release()
    camera_handle = None


def grab_frame():
    global current_frame
    if camera_handle is None:
        raise RuntimeError('Camera is not open.')
    best_frame = None
    best_brightness = -1.0
    for _ in range(6):
        frame = read_rgb_frame(camera_handle)
        brightness = float(frame.mean())
        if brightness > best_brightness:
            best_frame = frame
            best_brightness = brightness
        time.sleep(0.03)
    current_frame = best_frame
    render_image(raw_preview, current_frame)
    set_status(f'frame captured (mean={best_brightness:.1f})')


def open_serial_device(port, baudrate, timeout=0.4):
    ser = serial.Serial(port, baudrate=baudrate, timeout=timeout)
    ser.reset_input_buffer()
    ser.reset_output_buffer()
    return ser


def close_serial_device():
    global serial_handle
    if serial_handle is not None and serial_handle.is_open:
        serial_handle.close()
    serial_handle = None


def send_json(payload):
    if serial_handle is None or not serial_handle.is_open:
        raise RuntimeError('Serial port is not open.')
    line = json.dumps(payload, separators=(',', ':')) + '\n'
    serial_handle.write(line.encode('utf-8'))
    serial_handle.flush()
    time.sleep(0.12)
    waiting = serial_handle.in_waiting
    response = ''
    if waiting:
        response = serial_handle.read(waiting).decode('utf-8', errors='replace')
    return line.strip(), response


def stop_now():
    sent, response = send_json({'T': 1, 'L': 0.0, 'R': 0.0})
    log(f'STOP {sent}')
    if response:
        log(f'RECV {response}')


def on_click_wrap(fn):
    def wrapped(_):
        try:
            fn()
        except Exception as exc:
            set_status(f'error: {exc}')
            log(f'ERROR: {exc}')
    return wrapped


In [4]:
def build_analysis_from_points(frame_rgb, mask, x_near_global, y_near_global, x_far_global, y_far_global, mode):
    height, width = frame_rgb.shape[:2]
    image_center_x = width / 2.0
    offset_norm = (x_near_global - image_center_x) / (width / 2.0)
    heading_norm = (x_far_global - x_near_global) / (width / 2.0)
    steering = float(np.clip(k_offset.value * offset_norm + k_heading.value * heading_norm, -1.0, 1.0))

    overlay = frame_rgb.copy()
    cv2.line(overlay, (int(image_center_x), 0), (int(image_center_x), height - 1), (255, 0, 0), 2)
    cv2.circle(overlay, (int(x_near_global), int(y_near_global)), 8, (0, 255, 0), -1)
    cv2.circle(overlay, (int(x_far_global), int(y_far_global)), 8, (0, 255, 0), -1)
    cv2.line(overlay, (int(x_near_global), int(y_near_global)), (int(x_far_global), int(y_far_global)), (0, 255, 0), 3)
    cv2.putText(overlay, mode, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 0), 2, cv2.LINE_AA)

    mask_rgb = np.stack([mask] * 3, axis=-1)
    return {
        'overlay': overlay,
        'mask_rgb': mask_rgb,
        'offset_norm': float(offset_norm),
        'heading_norm': float(heading_norm),
        'steering': steering,
        'x_near': float(x_near_global),
        'y_near': float(y_near_global),
        'x_far': float(x_far_global),
        'y_far': float(y_far_global),
        'mode': mode,
    }


def analyze_green_centerline(frame_rgb):
    height, width = frame_rgb.shape[:2]
    crop_start = int(height * crop_top_ratio.value)
    roi = frame_rgb[crop_start:, :]

    hsv = cv2.cvtColor(roi, cv2.COLOR_RGB2HSV)
    lower = np.array([h_min.value, s_min.value, v_min.value], dtype=np.uint8)
    upper = np.array([h_max.value, s_max.value, v_max.value], dtype=np.uint8)
    mask = cv2.inRange(hsv, lower, upper)

    if blur_kernel.value > 1:
        k = int(blur_kernel.value)
        if k % 2 == 0:
            k += 1
        mask = cv2.GaussianBlur(mask, (k, k), 0)
        _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    if morph_kernel.value > 1:
        kernel = np.ones((int(morph_kernel.value), int(morph_kernel.value)), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    row_near = int(mask.shape[0] * near_row_ratio.value)
    row_far = int(mask.shape[0] * far_row_ratio.value)

    def row_center(row_index):
        row_index = max(0, min(mask.shape[0] - 1, row_index))
        xs = np.flatnonzero(mask[row_index] > 0)
        if xs.size == 0:
            return None
        return float(xs.mean())

    x_near = row_center(row_near)
    x_far = row_center(row_far)
    if x_near is None or x_far is None:
        raise RuntimeError('Could not find the green center line. Adjust HSV or camera framing.')

    x_near_global = x_near
    x_far_global = x_far
    y_near_global = crop_start + row_near
    y_far_global = crop_start + row_far
    return build_analysis_from_points(frame_rgb, mask, x_near_global, y_near_global, x_far_global, y_far_global, 'auto')


def analyze_manual_line(frame_rgb):
    height, width = frame_rgb.shape[:2]
    x_near_global = float(np.clip(manual_x_near.value, 0, width - 1))
    y_near_global = float(np.clip(manual_y_near.value, 0, height - 1))
    x_far_global = float(np.clip(manual_x_far.value, 0, width - 1))
    y_far_global = float(np.clip(manual_y_far.value, 0, height - 1))
    mask = np.zeros((height, width), dtype=np.uint8)
    return build_analysis_from_points(frame_rgb, mask, x_near_global, y_near_global, x_far_global, y_far_global, 'manual')


def analyze_current_frame():
    global last_analysis
    if current_frame is None:
        raise RuntimeError('Capture a frame before analyzing.')
    auto_analysis = analyze_green_centerline(current_frame)
    if use_manual_points.value:
        last_analysis = analyze_manual_line(current_frame)
        mode_text = 'manual'
    else:
        last_analysis = auto_analysis
        mode_text = 'auto'

    auto_x_near.value = int(round(auto_analysis['x_near']))
    auto_y_near.value = int(round(auto_analysis['y_near']))
    auto_x_far.value = int(round(auto_analysis['x_far']))
    auto_y_far.value = int(round(auto_analysis['y_far']))
    render_image(mask_preview, last_analysis['mask_rgb'])
    render_image(overlay_preview, last_analysis['overlay'])
    steering_output.value = last_analysis['steering']
    metrics.value = (
        f"<b>Mode:</b> {mode_text} &nbsp; "
        f"<b>Offset:</b> {last_analysis['offset_norm']:+.3f} &nbsp; "
        f"<b>Heading:</b> {last_analysis['heading_norm']:+.3f} &nbsp; "
        f"<b>Steering:</b> {last_analysis['steering']:+.3f}"
    )
    set_status('analysis ready')


def pick_two_points_from_image():
    if current_frame is None:
        raise RuntimeError('Capture a frame before picking points.')

    plt.figure(figsize=(10, 6))
    plt.imshow(current_frame)
    plt.title('Click 2 points on the road centerline: first near point, then far point')
    plt.axis('on')
    pts = plt.ginput(2, timeout=-1)
    plt.close()

    if len(pts) != 2:
        raise RuntimeError('Expected exactly 2 clicks.')

    (x1, y1), (x2, y2) = pts
    if y1 >= y2:
        near_x, near_y, far_x, far_y = x1, y1, x2, y2
    else:
        near_x, near_y, far_x, far_y = x2, y2, x1, y1

    manual_x_near.value = int(round(near_x))
    manual_y_near.value = int(round(near_y))
    manual_x_far.value = int(round(far_x))
    manual_y_far.value = int(round(far_y))
    use_manual_points.value = True
    set_status(f'manual points set from clicks: near=({manual_x_near.value}, {manual_y_near.value}), far=({manual_x_far.value}, {manual_y_far.value})')


def copy_auto_points_to_manual():
    manual_x_near.value = auto_x_near.value
    manual_y_near.value = auto_y_near.value
    manual_x_far.value = auto_x_far.value
    manual_y_far.value = auto_y_far.value
    set_status('copied auto points to manual controls')


def drive_once_from_analysis():
    if last_analysis is None:
        raise RuntimeError('Run analysis before driving.')
    steering = float(last_analysis['steering'])
    throttle = float(base_speed.value)
    mix = float(turn_mix.value)
    left = float(np.clip(throttle + mix * steering, -1.0, 1.0))
    right = float(np.clip(throttle - mix * steering, -1.0, 1.0))
    sent, response = send_json({'T': 1, 'L': left, 'R': right})
    log(f'DRIVE {sent}')
    if response:
        log(f'RECV {response}')
    time.sleep(step_time.value)
    stop_now()
    set_status(f'drove one step with steering={steering:+.3f}')


In [5]:
port_options = available_ports() or ['/dev/ttyTHS1', '/dev/ttyTHS2']
default_port = '/dev/ttyTHS1' if '/dev/ttyTHS1' in port_options else port_options[0]

camera_source_dropdown = widgets.Dropdown(options=['usb', 'csi'], value='usb', description='Source')
sensor_id = widgets.IntText(value=0, description='Sensor')
device_index = widgets.IntText(value=0, description='Device')
frame_width = widgets.IntText(value=1280, description='Width')
frame_height = widgets.IntText(value=720, description='Height')
warmup_frames = widgets.IntText(value=12, description='Warmup')

port_dropdown = widgets.Dropdown(options=port_options, value=default_port, description='Port')
baud_input = widgets.IntText(value=115200, description='Baud')
base_speed = widgets.FloatSlider(value=0.22, min=0.05, max=0.50, step=0.01, description='Speed')
turn_mix = widgets.FloatSlider(value=0.45, min=0.10, max=1.20, step=0.05, description='Turn Mix')
step_time = widgets.FloatSlider(value=0.20, min=0.05, max=0.60, step=0.01, description='Step s')

crop_top_ratio = widgets.FloatSlider(value=0.35, min=0.0, max=0.8, step=0.05, description='Crop Top')
near_row_ratio = widgets.FloatSlider(value=0.82, min=0.50, max=0.98, step=0.02, description='Near Row')
far_row_ratio = widgets.FloatSlider(value=0.42, min=0.05, max=0.80, step=0.02, description='Far Row')
h_min = widgets.IntSlider(value=35, min=0, max=179, step=1, description='H min')
h_max = widgets.IntSlider(value=95, min=0, max=179, step=1, description='H max')
s_min = widgets.IntSlider(value=40, min=0, max=255, step=1, description='S min')
s_max = widgets.IntSlider(value=255, min=0, max=255, step=1, description='S max')
v_min = widgets.IntSlider(value=40, min=0, max=255, step=1, description='V min')
v_max = widgets.IntSlider(value=255, min=0, max=255, step=1, description='V max')
blur_kernel = widgets.IntSlider(value=5, min=1, max=15, step=2, description='Blur')
morph_kernel = widgets.IntSlider(value=5, min=1, max=15, step=2, description='Morph')
k_offset = widgets.FloatSlider(value=0.85, min=0.0, max=2.0, step=0.05, description='K offset')
k_heading = widgets.FloatSlider(value=0.60, min=0.0, max=2.0, step=0.05, description='K heading')
use_manual_points = widgets.Checkbox(value=False, description='Use Manual Points')
copy_points_button = widgets.Button(description='Copy Auto -> Manual')
pick_points_button = widgets.Button(description='Pick 2 Points', button_style='info')
auto_x_near = widgets.IntText(value=0, description='Auto Xn', disabled=True)
auto_y_near = widgets.IntText(value=0, description='Auto Yn', disabled=True)
auto_x_far = widgets.IntText(value=0, description='Auto Xf', disabled=True)
auto_y_far = widgets.IntText(value=0, description='Auto Yf', disabled=True)
manual_x_near = widgets.IntSlider(value=420, min=0, max=1279, step=1, description='Near X')
manual_y_near = widgets.IntSlider(value=580, min=0, max=719, step=1, description='Near Y')
manual_x_far = widgets.IntSlider(value=520, min=0, max=1279, step=1, description='Far X')
manual_y_far = widgets.IntSlider(value=300, min=0, max=719, step=1, description='Far Y')

open_camera_button = widgets.Button(description='Open Camera', button_style='success')
close_camera_button = widgets.Button(description='Close Camera')
grab_button = widgets.Button(description='Grab Frame', button_style='info')
analyze_button = widgets.Button(description='Analyze Frame', button_style='warning')

open_port_button = widgets.Button(description='Open Port', button_style='success')
close_port_button = widgets.Button(description='Close Port')
stop_button = widgets.Button(description='Stop', button_style='danger')
drive_once_button = widgets.Button(description='Drive One Step', button_style='primary')

steering_output = widgets.FloatText(value=0.0, description='Steer', disabled=True)
metrics = widgets.HTML(value='<b>Offset:</b> -- <b>Heading:</b> -- <b>Steering:</b> --')
status = widgets.HTML(value='<b>Status:</b> idle')
log_output = widgets.Output(layout={'border': '1px solid #444', 'max_height': '220px', 'overflow_y': 'auto'})
raw_preview = widgets.Image(format='jpeg', width=420, height=236)
mask_preview = widgets.Image(format='jpeg', width=420, height=236)
overlay_preview = widgets.Image(format='jpeg', width=420, height=236)

open_camera_button.on_click(on_click_wrap(lambda: open_selected_camera()))
close_camera_button.on_click(on_click_wrap(lambda: [close_camera(), set_status('camera closed')]))
grab_button.on_click(on_click_wrap(lambda: grab_frame()))
analyze_button.on_click(on_click_wrap(lambda: analyze_current_frame()))
copy_points_button.on_click(on_click_wrap(lambda: copy_auto_points_to_manual()))
pick_points_button.on_click(on_click_wrap(lambda: pick_two_points_from_image()))

open_port_button.on_click(on_click_wrap(lambda: [close_serial_device(), globals().__setitem__('serial_handle', open_serial_device(port_dropdown.value, baud_input.value)), set_status(f'serial open on {port_dropdown.value}')]))
close_port_button.on_click(on_click_wrap(lambda: [close_serial_device(), set_status('serial closed')]))
stop_button.on_click(on_click_wrap(lambda: [stop_now(), set_status('stop sent')]))
drive_once_button.on_click(on_click_wrap(lambda: drive_once_from_analysis()))

panel = widgets.VBox([
    widgets.HTML(value=f"<b>Video devices:</b> {' '.join(video_devices) if video_devices else 'none found'}"),
    widgets.HBox([camera_source_dropdown, sensor_id, device_index]),
    widgets.HBox([frame_width, frame_height, warmup_frames]),
    widgets.HBox([open_camera_button, close_camera_button, grab_button, analyze_button]),
    widgets.HTML(value=f"<b>Serial ports:</b> {' '.join(port_options) if port_options else 'none found'}"),
    widgets.HBox([port_dropdown, baud_input]),
    widgets.HBox([base_speed, turn_mix, step_time]),
    widgets.HBox([open_port_button, close_port_button, stop_button, drive_once_button]),
    widgets.HTML(value='<h4>Detection Tuning</h4>'),
    crop_top_ratio,
    widgets.HBox([near_row_ratio, far_row_ratio]),
    widgets.HBox([h_min, h_max]),
    widgets.HBox([s_min, s_max]),
    widgets.HBox([v_min, v_max]),
    widgets.HBox([blur_kernel, morph_kernel]),
    widgets.HBox([k_offset, k_heading, steering_output]),
    widgets.HTML(value='<h4>Points</h4>'),
    widgets.HBox([use_manual_points, copy_points_button, pick_points_button]),
    widgets.HBox([auto_x_near, auto_y_near, auto_x_far, auto_y_far]),
    widgets.HBox([manual_x_near, manual_y_near]),
    widgets.HBox([manual_x_far, manual_y_far]),
    metrics,
    status,
    widgets.HBox([raw_preview, mask_preview, overlay_preview]),
    log_output,
])

display(panel)


## Suggested Tuning Order

1. Open the camera and grab a frame.
2. Adjust HSV sliders until the mask mostly captures the green center tape.
3. Check the two detected points in the overlay.
4. Tune `K offset` and `K heading` until the steering value feels reasonable.
5. Open the serial port.
6. Use `Drive One Step` only while the rover is on a stand or moving slowly.

The steering sign convention here is:

- negative = steer left
- positive = steer right
